# False Positive Reduction using Random Forest Confidence Scoring

This notebook applies a **Random Forest model** as a confidence-based filtering layer
to reduce false positives in web vulnerability scanner outputs.

**Key idea:** ML is used to assign FP likelihood scores, not hard classifications.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

## Load Cleaned Dataset

In [ ]:
# Note: the original 32MB dataset (collected from authorized-but-confidential

# scans) is not shipped in this repo. This loads a small synthetic sample
# with fabricated project/URL data that exercises the same schema instead.
df = pd.read_csv("sample_dataset.csv")
print("Total rows:", len(df))
df.head()

## Feature Engineering

In [ ]:
df["name_norm"] = df["name"].str.lower().str.strip()
df["project_name_norm"] = df["project_name"].str.lower().str.strip()

df.head()

In [ ]:
header_vulns = [
    "missing_security_header",
    "missing anti-clickjacking header",
    "content security policy (csp) header not set",
    "missing content-security-policy header",
    "strict-transport-security header not set",
    "strict-transport-security missing max-age (non-compliant with spec)",
    "missing hsts header",
    "x-content-type-options header missing",
    "missing x-content-type-options header",
    "missing x-frame-options header",
    "multiple x-frame-options header entries",
    "missing x-xss-protection header",
    "missing referrer-policy header",
    "x-aspnet-version response header",
    "server leaks information via \"x-powered-by\" http response header field(s)",
    "server leaks version information via \"server\" http response header field",
    "permissive cors policy",
    "csp: notices",
    "csp: wildcard directive",
    "csp: style-src unsafe-inline",
    "csp: script-src unsafe-inline",
    "csp: script-src unsafe-eval",
    "csp: failure to define directive with no fallback"
]

df["is_header_issue"] = df["name_norm"].isin(header_vulns).astype(int)

In [ ]:
injection_vulns = [
    "sql injection",
    "sql injection - mysql",
    "sql injection - mysql (time based)",
    "sql injection - sqlite (time based)",
    "sql injection - mssql (time based)",
    "cross site scripting (reflected)",
    "cross site scripting (dom based)",
    "cross site scripting (persistent)",
    "xslt injection",
    "format string error",
    "session id in url rewrite"
]

df["is_injection"] = df["name_norm"].isin(injection_vulns).astype(int)

## Identify Real-world Applications

In [ ]:
vulnerable_apps = {
    "owasp juice shop",
    "owasp juice shop - e-commerce vulnerabilities",
    "bwapp",
    "bwapp (buggy web application)",
    "dvwa",
    "local dvwa",
    "dvwa sqli",
    "sql injection test project",
    "vulnerable app",
    "demo",
    "portswigger web security academy labs",
    "google's xss testing ground",
    "google gruyere – vulnerable app",
    "ibm banking app vulnerabilities",
    "ibm banking app vulnerabilities (testfire)",
    "classic web vulnerabilities",
    "classic web vulnerabilities (testphp)"
}

df["is_real_world"] = (~df["project_name_norm"].isin(vulnerable_apps)).astype(int)

In [ ]:
df.head()
df.to_csv('dataset_after_feature_engineering.csv', index=False)
df.head()

## Pseudo-labeling (Weak Supervision)

In [ ]:
df["likely_fp"] = 0

df.loc[
    (df["is_real_world"] == 1) &
    (df["is_header_issue"] == 1) &
    (df["has_evidence"] == 0) &
    (df["severity"].isin(["low", "medium"])),
    "likely_fp"
] = 1

df["likely_fp"].value_counts(normalize=True)

In [ ]:
df.head()

## Random Forest as FP Confidence Model

In [ ]:
real_df = df[df["is_real_world"] == 1]

features = [
    "has_evidence",
    "is_header_issue",
    "is_injection"
]

X = real_df[features]
y = real_df["likely_fp"]

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

## False Positive Confidence Scores

In [ ]:
real_df["fp_confidence"] = rf.predict_proba(X)[:, 1]
real_df[["name", "severity", "fp_confidence"]].head()

## Confidence Distribution

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import FileLink

plt.figure(figsize=(8, 5))
plt.hist(real_df["fp_confidence"], bins=30)
plt.xlabel("False Positive Confidence")
plt.ylabel("Number of Findings")
plt.title("FP Confidence Distribution (Random Forest)")
plt.tight_layout()

# Save figure
plt.savefig("fp_confidence_distribution_rf.png", dpi=300, bbox_inches="tight")
plt.show()

# Download link
FileLink("fp_confidence_distribution_rf.png")


## Threshold-based Filtering

In [ ]:
threshold = 0.7

real_df["fp_filtered"] = (real_df["fp_confidence"] >= threshold).astype(int)

total_alerts = len(real_df)
filtered_alerts = real_df["fp_filtered"].sum()

print(f"Total alerts: {total_alerts}")
print(f"Filtered as likely FP: {filtered_alerts}")
print(f"FP Reduction: {(filtered_alerts / total_alerts) * 100:.2f}%")

## Threshold Sensitivity Analysis

In [ ]:
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    filtered = (real_df["fp_confidence"] >= t).sum()
    print(f"Threshold {t}: {filtered} alerts filtered ({(filtered/total_alerts)*100:.2f}%)")

In [ ]:
import joblib

joblib.dump(rf, "fp_confidence_random_forest.pkl")

print("✅ Model saved successfully")


In [ ]:
before_count = len(real_df)
after_count = len(real_df[real_df["fp_filtered"] == 0])

plt.figure(figsize=(6,4))
plt.bar(["Before ML", "After ML"], [before_count, after_count])
plt.ylabel("Number of Alerts")
plt.title("Alert Reduction After ML (Random Forest)")

for i, v in enumerate([before_count, after_count]):
    plt.text(i, v + 10, str(v), ha="center", fontweight="bold")

plt.tight_layout()

# Save figure
plt.savefig("alert_reduction_rf.png", dpi=300, bbox_inches="tight")
plt.show()

# Download link
FileLink("alert_reduction_rf.png")


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(real_df["fp_confidence"], bins=30)
plt.axvline(threshold, linestyle="--")
plt.xlabel("False Positive Confidence")
plt.ylabel("Count")
plt.title("FP Confidence Distribution (Random Forest)")
plt.tight_layout()
plt.show()


In [ ]:
severity_before = real_df["severity"].value_counts()
severity_after = real_df[real_df["fp_filtered"] == 0]["severity"].value_counts()

severity_df = (
    pd.DataFrame({
        "Before ML": severity_before,
        "After ML": severity_after
    })
    .fillna(0)
)

severity_df = severity_df.reindex(["high", "medium", "low"])

severity_df.plot(kind="bar", figsize=(7,4))
plt.ylabel("Number of Alerts")
plt.title("Severity Preservation Analysis (Random Forest)")
plt.tight_layout()

# Save figure
plt.savefig("severity_preservation_rf.png", dpi=300, bbox_inches="tight")
plt.show()

# Download link
FileLink("severity_preservation_rf.png")


In [ ]:
inj_stats = pd.crosstab(
    real_df["is_injection"],
    real_df["fp_filtered"]
)

inj_stats.plot(kind="bar", figsize=(6,4))
plt.xlabel("Injection Vulnerability (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.title("Injection Vulnerability Preservation (Random Forest)")
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import display, HTML

def display_with_scroll(df, title="", height=400):
    display(HTML(f"<h4>{title}</h4>"))
    display(HTML(
        f"<div style='max-height:{height}px; overflow:auto; border:1px solid #ccc;'>"
        f"{df.to_html(index=False)}"
        f"</div>"
    ))


In [ ]:
before_df = real_df
after_df = real_df[real_df["fp_filtered"] == 0]

comparison_df = pd.DataFrame({
    "Before ML": before_df["name"].value_counts(),
    "After ML": after_df["name"].value_counts()
}).fillna(0).astype(int)

display_with_scroll(
    comparison_df.reset_index().rename(columns={"index": "Vulnerability Name"}),
    title="Before vs After ML – Vulnerability Counts (Random Forest)",
    height=400
)
